In [89]:
# This code extracts the sequence and chain informations according to the assigned tutorials

from biopandas.pdb import PandasPdb
import numpy as np

import os

# Specify the directory where you want to save the files
output_directory = "../Outputs/data_manipulation"

# Create the directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

In [90]:
# Initialize a new PandasPdb object and fetch the PDB file from rcsb.org
# ppdb = PandasPdb().fetch_pdb('1crn')
ppdb = PandasPdb().fetch_pdb('2C0K')

# Display the type of information in each dataframe
for df_name in ppdb.df:
    print(f"Dataframe: {df_name}")
    print(ppdb.df[df_name].info())
    print(ppdb.df[df_name].head(), "\n")

# Optionally, display the column names for each dataframe
for df_name in ppdb.df:
    print(f"Columns in {df_name}:")
    print(ppdb.df[df_name].columns, "\n")

Dataframe: ATOM
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2461 entries, 0 to 2460
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   record_name     2461 non-null   object 
 1   atom_number     2461 non-null   int64  
 2   blank_1         2461 non-null   object 
 3   atom_name       2461 non-null   object 
 4   alt_loc         2461 non-null   object 
 5   residue_name    2461 non-null   object 
 6   blank_2         2461 non-null   object 
 7   chain_id        2461 non-null   object 
 8   residue_number  2461 non-null   int64  
 9   insertion       2461 non-null   object 
 10  blank_3         2461 non-null   object 
 11  x_coord         2461 non-null   float64
 12  y_coord         2461 non-null   float64
 13  z_coord         2461 non-null   float64
 14  occupancy       2461 non-null   float64
 15  b_factor        2461 non-null   float64
 16  blank_4         2461 non-null   object 
 17  segment_id      2

In [91]:
df_atoms = ppdb.df["ATOM"]
df_atoms

,record_name,atom_number,blank_1,atom_name,alt_loc,residue_name,blank_2,chain_id,residue_number,insertion,...,x_coord,y_coord,z_coord,occupancy,b_factor,blank_4,segment_id,element_symbol,charge,line_idx
0,ATOM,1,,N,,MET,,A,1,,...,10.263,-7.566,-4.747,1.0,47.36,,,N,NaN,445
1,ATOM,2,,CA,,MET,,A,1,,...,9.077,-7.905,-5.617,1.0,47.69,,,C,NaN,446
2,ATOM,3,,C,,MET,,A,1,,...,9.155,-9.333,-6.212,1.0,47.89,,,C,NaN,447
3,ATOM,4,,O,,MET,,A,1,,...,10.028,-9.649,-7.048,1.0,48.03,,,O,NaN,448
4,ATOM,5,,CB,,MET,,A,1,,...,8.869,-6.852,-6.731,1.0,47.38,,,C,NaN,449
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2456,ATOM,2458,,CB,,LYS,,B,149,,...,12.860,21.471,10.127,1.0,64.66,,,C,NaN,2902
2457,ATOM,2459,,CG,,LYS,,B,149,,...,14.018,20.896,10.955,1.0,63.42,,,C,NaN,2903
2458,ATOM,2460,,CD,,LYS,,B,149,,...,13.538,19.754,11.844,1.0,62.80,,,C,NaN,2904
2459,ATOM,2461,,CE,,LYS,,B,149,,...,14.473,18.565,11.804,1.0,61.05,,,C,NaN,2905


In [92]:
df_others = ppdb.df["OTHERS"]
df_others

,record_name,entry,line_idx
0,HEADER,OXYGEN TRANSPORT 05...,0
1,TITLE,THE STRUCTURE OF HEMOGLOBIN FROM THE BOTFL...,1
2,COMPND,MOL_ID: 1;,2
3,COMPND,2 MOLECULE: HEMOGLOBIN;,3
4,COMPND,"3 CHAIN: A, B",4
...,...,...,...
538,CONECT,2551 2550 2552 2553,3095
539,CONECT,2552 2551 2553,3096
540,CONECT,2553 2551 2552,3097
541,MASTER,338 0 4 20 0 0 10 9 2...,3098


TUTORIAL 1:
Extract sequence from a PDB ATOM record and save to a fasta file format

In [93]:
# Extract the HEADER part of the PDB file
header_entry = df_others[df_others['record_name'] == 'HEADER']['entry'].values
print(header_entry)

# Pull the last string. Corresponds to the code of the protein
header_entry = header_entry[0]
code_name = header_entry.split()[-1]

print("\nCode:",code_name)

['    OXYGEN TRANSPORT                        05-SEP-05   2C0K']

Code: 2C0K


In [94]:
# Extract the COMPOUND part of the PDB file
compnd_entry = df_others[df_others['record_name'] == 'COMPND']['entry'].values

print(compnd_entry)

molecule_name = None
chains = None

# Iterate over each line in the array
# This section of code assumes there is only one set of chain and one sequence in the protein
# For multiple set of chains and sequences, this code can be modified by adding loops
for line in compnd_entry:
    # Check if the line contains the molecule name
    if 'MOLECULE' in line:
        molecule_name = line.split(':')[1].strip()
        molecule_name = molecule_name.replace(";","")
    # Check if the line contains chain information
    if 'CHAIN' in line:
        chains = (line.split(':')[1].strip())
        chains = chains.replace(";","")

print("\nMolecule Name: ", molecule_name)
print("Chains: ", chains)


['    MOL_ID: 1;' '   2 MOLECULE: HEMOGLOBIN;' '   3 CHAIN: A, B']

Molecule Name:  HEMOGLOBIN
Chains:  A, B


In [95]:
# Extract the SOURCE part of the PDB file
source_entry = df_others[df_others['record_name'] == 'SOURCE']['entry'].values
print(source_entry)

scientific_name = None
taxid_name = None

# Iterate over each line in the array
for line in source_entry:
    # Check if the line contains the scientific name
    if 'ORGANISM_SCIENTIFIC' in line:
        scientific_name = line.split(':')[1].strip()
        scientific_name = scientific_name.replace(";","")
    if 'ORGANISM_TAXID' in line:
        taxid_name = line.split(':')[1].strip()
        taxid_name = taxid_name.replace(";","")
        taxid_name = "(" + taxid_name + ")" 

print("\nOrganism Name:" + scientific_name)
print("Tax ID:" + taxid_name)

['    MOL_ID: 1;' '   2 ORGANISM_SCIENTIFIC: GASTEROPHILUS INTESTINALIS;'
 '   3 ORGANISM_TAXID: 84525']

Organism Name:GASTEROPHILUS INTESTINALIS
Tax ID:(84525)


In [96]:
# Extract each residue name and convert them to one letter abbreviations of amino acids
amino_acid_sequence = ppdb.amino3to1().residue_name

# print("Amino Acid Sequence: ")
# print(''.join(amino_acid_sequence))

amino_acid_sequence2 = ppdb.amino3to1()
print(amino_acid_sequence2)

amino_acid_sequence = ''.join(amino_acid_sequence)
print(amino_acid_sequence)

     chain_id residue_name
0           A            M
8           A            N
16          A            S
22          A            E
31          A            E
...       ...          ...
2425        B            A
2430        B            E
2439        B            M
2447        B            A
2452        B            K

[299 rows x 2 columns]
MNSEEVNDIKRTWEVVAAKMTEAGVEMLKRYFKKYPHNLNHFPWFKEIPFDDLPENARFKTHGTRILRQVDEGVKALSVDFGDKKFDDVWKKLAQTHHEKKVERRSYNELKDIIIEVVCSCVKLNEKQVHAYHKFFDRAYDIAFAEMAKMMNSEEVNDIKRTWEVVAAKMTEAGVEMLKRYFKKYPHNLNHFPWFKEIPFDDLPENARFKTHGTRILRQVDEGVKALSVDFGDKKFDDVWKKLAQTHHEKKVERRSYNELKDIIIEVVCSCVKLNEKQVHAYHKFFDRAYDIAFAEMAK


In [97]:
# Order:
# code_name
# chains
# molecule_name
# scientific_name
# taxid_name
# amino_acid_sequence

first_line = code_name + "|" + "Chains " + chains + "|" + molecule_name + "|" + scientific_name + " " + taxid_name
second_line = amino_acid_sequence

In [98]:
# Write sequences to a FASTA file

file_name = os.path.join(output_directory, f"tutorial_1.fasta")

with open(file_name, "w") as fasta_file:
    fasta_file.write(f">{first_line}\n")
    fasta_file.write(f"{second_line}\n")

TUTORIAL 2:
Extract the full sequence from the SEQRES record and save it to a fasta file format. 

TUTORIAL 3:
Pick a PDB with multiple chains. Extract chains and save each chain to an individual PDB.

In [99]:
seqres_entry = df_others[df_others['record_name'] == 'SEQRES']['entry'].values
seqres_entry

array(['   1 A  151  MET ASN SER GLU GLU VAL ASN ASP ILE LYS ARG THR TRP',
       '   2 A  151  GLU VAL VAL ALA ALA LYS MET THR GLU ALA GLY VAL GLU',
       '   3 A  151  MET LEU LYS ARG TYR PHE LYS LYS TYR PRO HIS ASN LEU',
       '   4 A  151  ASN HIS PHE PRO TRP PHE LYS GLU ILE PRO PHE ASP ASP',
       '   5 A  151  LEU PRO GLU ASN ALA ARG PHE LYS THR HIS GLY THR ARG',
       '   6 A  151  ILE LEU ARG GLN VAL ASP GLU GLY VAL LYS ALA LEU SER',
       '   7 A  151  VAL ASP PHE GLY ASP LYS LYS PHE ASP ASP VAL TRP LYS',
       '   8 A  151  LYS LEU ALA GLN THR HIS HIS GLU LYS LYS VAL GLU ARG',
       '   9 A  151  ARG SER TYR ASN GLU LEU LYS ASP ILE ILE ILE GLU VAL',
       '  10 A  151  VAL CYS SER CYS VAL LYS LEU ASN GLU LYS GLN VAL HIS',
       '  11 A  151  ALA TYR HIS LYS PHE PHE ASP ARG ALA TYR ASP ILE ALA',
       '  12 A  151  PHE ALA GLU MET ALA LYS MET GLY',
       '   1 B  151  MET ASN SER GLU GLU VAL ASN ASP ILE LYS ARG THR TRP',
       '   2 B  151  GLU VAL VAL ALA ALA LYS 

In [100]:
# Initialize an empty dictionary
result_dict = {}

# Process each string in the array
for item in seqres_entry:
    # Split the string into words
    words = item.split()
    
    # Extract the second letter to get the chain identifier "A"
    second_letter = words[1]
    
    # Extract the substring after the '46', which starts from the 4th word
    substring = ' '.join(words[3:])
    
    # Append the substring to the corresponding key in the dictionary
    if second_letter in result_dict:
        result_dict[second_letter].append(substring)
    else:
        result_dict[second_letter] = [substring]

print(result_dict)

{'A': ['MET ASN SER GLU GLU VAL ASN ASP ILE LYS ARG THR TRP', 'GLU VAL VAL ALA ALA LYS MET THR GLU ALA GLY VAL GLU', 'MET LEU LYS ARG TYR PHE LYS LYS TYR PRO HIS ASN LEU', 'ASN HIS PHE PRO TRP PHE LYS GLU ILE PRO PHE ASP ASP', 'LEU PRO GLU ASN ALA ARG PHE LYS THR HIS GLY THR ARG', 'ILE LEU ARG GLN VAL ASP GLU GLY VAL LYS ALA LEU SER', 'VAL ASP PHE GLY ASP LYS LYS PHE ASP ASP VAL TRP LYS', 'LYS LEU ALA GLN THR HIS HIS GLU LYS LYS VAL GLU ARG', 'ARG SER TYR ASN GLU LEU LYS ASP ILE ILE ILE GLU VAL', 'VAL CYS SER CYS VAL LYS LEU ASN GLU LYS GLN VAL HIS', 'ALA TYR HIS LYS PHE PHE ASP ARG ALA TYR ASP ILE ALA', 'PHE ALA GLU MET ALA LYS MET GLY'], 'B': ['MET ASN SER GLU GLU VAL ASN ASP ILE LYS ARG THR TRP', 'GLU VAL VAL ALA ALA LYS MET THR GLU ALA GLY VAL GLU', 'MET LEU LYS ARG TYR PHE LYS LYS TYR PRO HIS ASN LEU', 'ASN HIS PHE PRO TRP PHE LYS GLU ILE PRO PHE ASP ASP', 'LEU PRO GLU ASN ALA ARG PHE LYS THR HIS GLY THR ARG', 'ILE LEU ARG GLN VAL ASP GLU GLY VAL LYS ALA LEU SER', 'VAL ASP PHE GLY

In [101]:
# Initialize a new dictionary to store the merged strings
merged_dict = {}

# Iterate through each key in the dictionary
for key, strings in result_dict.items():
    # Merge all strings in the list into one string
    merged_string = ' '.join(strings)
    # Add the merged string to the new dictionary
    merged_dict[key] = merged_string

print(merged_dict)

{'A': 'MET ASN SER GLU GLU VAL ASN ASP ILE LYS ARG THR TRP GLU VAL VAL ALA ALA LYS MET THR GLU ALA GLY VAL GLU MET LEU LYS ARG TYR PHE LYS LYS TYR PRO HIS ASN LEU ASN HIS PHE PRO TRP PHE LYS GLU ILE PRO PHE ASP ASP LEU PRO GLU ASN ALA ARG PHE LYS THR HIS GLY THR ARG ILE LEU ARG GLN VAL ASP GLU GLY VAL LYS ALA LEU SER VAL ASP PHE GLY ASP LYS LYS PHE ASP ASP VAL TRP LYS LYS LEU ALA GLN THR HIS HIS GLU LYS LYS VAL GLU ARG ARG SER TYR ASN GLU LEU LYS ASP ILE ILE ILE GLU VAL VAL CYS SER CYS VAL LYS LEU ASN GLU LYS GLN VAL HIS ALA TYR HIS LYS PHE PHE ASP ARG ALA TYR ASP ILE ALA PHE ALA GLU MET ALA LYS MET GLY', 'B': 'MET ASN SER GLU GLU VAL ASN ASP ILE LYS ARG THR TRP GLU VAL VAL ALA ALA LYS MET THR GLU ALA GLY VAL GLU MET LEU LYS ARG TYR PHE LYS LYS TYR PRO HIS ASN LEU ASN HIS PHE PRO TRP PHE LYS GLU ILE PRO PHE ASP ASP LEU PRO GLU ASN ALA ARG PHE LYS THR HIS GLY THR ARG ILE LEU ARG GLN VAL ASP GLU GLY VAL LYS ALA LEU SER VAL ASP PHE GLY ASP LYS LYS PHE ASP ASP VAL TRP LYS LYS LEU ALA GLN T

In [102]:
file_name2 = os.path.join(output_directory, f"tutorial_2.fasta")

# Save the the all sequence at once
with open(file_name2, "w") as fasta_file:
    fasta_file.write(f">")
    for key in merged_dict:
        temp_string = merged_dict[key]
        fasta_file.write(f"{temp_string} ")

# # Save the sequence to a fasta file by each chain
# with open("output3.fasta", "w") as fasta_file:
#     fasta_file.write(f">")
#     for key in merged_dict:
#         temp_string = merged_dict[key]
#         fasta_file.write(f"Chain {key}: {{temp_string}\n")

# # Save the merged dictionary to a pdb file
# with open('tutorial_3.pdb', 'w') as pdb_file:
#     for chain, sequence in merged_dict.items():
#         pdb_file.write(f"Chain {chain}: {sequence}\n")

# Save the merged dictionary to separate pdb files
for chain, sequence in merged_dict.items():
    file_name3 = os.path.join(output_directory, f"tutorial_3_chain_{chain}.pdb")
    with open(file_name3, 'w') as pdb_file:
        pdb_file.write(f"Chain {chain}: {sequence}\n")